# 01 — Data Acquisition

**Objective**: Download and inspect the baseline datasets for the canopy height comparison study.

## Datasets

| Dataset | Source | Resolution | Purpose |
|---------|--------|-----------|---------|
| **ALS CHM** (Reis & Jackson) | [Zenodo](https://zenodo.org/records/7104044) | 1 m | Ground truth — 487 airborne LiDAR transects across the Brazilian Amazon |
| **Meta Global CHM** | [GEE Community Catalog](https://gee-community-catalog.org/projects/meta_trees/) | 1 m (exported at 10 m) | Global satellite-derived canopy height product |
| **GEDI L2A** | Google Earth Engine | 25 m footprint | Spaceborne LiDAR canopy height shots (rh98) for independent validation |

## Study Area

Northern Tocantins, Brazil (~-47.2°W, -7.5°S) — a cerrado-to-Amazon transition zone with canopy heights ranging from open grassland (0 m) to tall closed-canopy forest (>30 m).

In [ ]:
import os
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import rioxarray
import matplotlib.pyplot as plt
from pathlib import Path
import json

import sys
sys.path.insert(0, "..")
from src.config import GEE_PROJECT, AOI_PATH, DATA_DIR

# Initialize GEE
ee.Initialize(project=GEE_PROJECT)
print("GEE initialized")

## 1. Load Study Area

In [ ]:
# Load AOI as GeoDataFrame and ee.Geometry
gdf = gpd.read_file(AOI_PATH)
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

with open(AOI_PATH) as f:
    aoi_json = json.load(f)
coords = aoi_json["features"][0]["geometry"]["coordinates"][0][0]
ee_geom = ee.Geometry.Polygon(coords)

print(f"AOI bounds: {gdf.total_bounds}")
print(f"AOI area: ~{gdf.to_crs(epsg=32623).area.sum() / 1e6:.1f} km²")

gdf.plot(edgecolor="red", facecolor="none", figsize=(6, 6))
plt.title("AOI — Northern Tocantins, Brazil")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

## 2. Download Meta Global Canopy Height Map

The Meta Global Canopy Height Map provides wall-to-wall 1m canopy height estimates derived from high-resolution satellite imagery using self-supervised vision transformers trained on aerial LiDAR.

- **Asset**: `projects/sat-io/open-datasets/facebook/meta-canopy-height`
- **Resolution**: 1m native, exported here at 10m for manageability
- **NoData**: 255 (0 = valid, meaning no canopy)

In [ ]:
meta_chm_path = DATA_DIR / "meta_chm_10m.tif"

meta_ic = ee.ImageCollection(
    "projects/sat-io/open-datasets/facebook/meta-canopy-height"
).filterBounds(ee_geom)

print(f"Meta CHM tiles covering AOI: {meta_ic.size().getInfo()}")

meta_chm_img = meta_ic.mosaic().clip(ee_geom)

print("Exporting Meta CHM at 10m...")
geemap.ee_export_image(
    meta_chm_img,
    filename=str(meta_chm_path),
    scale=10,
    region=ee_geom,
    file_per_band=False,
)
print(f"Saved to {meta_chm_path}")

In [ ]:
# Fix nodata and inspect
ds = rioxarray.open_rasterio(meta_chm_path)
ds = ds.rio.write_nodata(255)
ds.rio.to_raster(str(meta_chm_path))

ds = rioxarray.open_rasterio(meta_chm_path)
print(f"Shape: {ds.shape}")
print(f"CRS: {ds.rio.crs}")
print(f"Resolution: {ds.rio.resolution()}")
print(f"Value range: {float(ds.min())} – {float(ds.max())} m")

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ds.sel(band=1).plot(ax=ax, cmap="YlGn", vmin=0, vmax=30,
                     cbar_kwargs={"label": "Canopy Height (m)"})
gdf.boundary.plot(ax=ax, edgecolor="red", linewidth=2)
ax.set_title("Meta Global Canopy Height Map (10m)")
plt.tight_layout()
plt.show()

## 3. Download GEDI L2A Canopy Height Shots

GEDI (Global Ecosystem Dynamics Investigation) provides spaceborne LiDAR measurements of canopy structure. We use `rh98` — the 98th percentile relative height — as a proxy for canopy top height.

- **Collection**: `LARSE/GEDI/GEDI02_A_002_MONTHLY`
- **Quality filter**: `quality_flag == 1` and `degrade_flag == 0`
- **Footprint**: ~25m diameter

In [ ]:
from src.data_processing.gee_utils import get_gedi_l2a

gedi_df = get_gedi_l2a(ee_geom)
gedi_path = DATA_DIR / "gedi_l2a_aoi.csv"
gedi_df.to_csv(gedi_path, index=False)
print(f"Saved {len(gedi_df)} shots to {gedi_path}")
gedi_df[["rh98", "rh75", "rh50"]].describe()

## 4. Overlay: Meta CHM + GEDI shots

Visual comparison showing the Meta CHM raster with GEDI shots overlaid. Good agreement in forested areas; divergence in transition zones suggests limitations of the global product in heterogeneous landscapes.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 9))

ds = rioxarray.open_rasterio(DATA_DIR / "meta_chm_10m.tif")
ds.sel(band=1).plot(ax=ax, cmap="YlGn", vmin=0, vmax=30,
                     cbar_kwargs={"label": "Meta CHM (m)"})

sc = ax.scatter(
    gedi_df["longitude"], gedi_df["latitude"],
    c=gedi_df["rh98"], cmap="hot_r", vmin=0, vmax=30,
    s=5, alpha=0.7, edgecolors="none",
)
plt.colorbar(sc, ax=ax, label="GEDI rh98 (m)", shrink=0.6)
gdf.boundary.plot(ax=ax, edgecolor="red", linewidth=2)

ax.set_title("Meta CHM (background) vs GEDI rh98 (points)")
plt.tight_layout()
plt.show()

## 5. Quick comparison: Meta CHM vs GEDI

Sampling the Meta CHM raster at GEDI shot locations gives an initial sense of product quality. The scatter plot below shows the relationship — note the spread and potential systematic bias.

In [ ]:
from rasterio.transform import rowcol
import rasterio
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

with rasterio.open(DATA_DIR / "meta_chm_10m.tif") as src:
    meta_vals = []
    for _, row in gedi_df.iterrows():
        try:
            r, c = rowcol(src.transform, row["longitude"], row["latitude"])
            val = src.read(1)[r, c]
            meta_vals.append(val if val != src.nodata else np.nan)
        except (IndexError, ValueError):
            meta_vals.append(np.nan)

gedi_compare = gedi_df.copy()
gedi_compare["meta_chm"] = meta_vals
gedi_compare = gedi_compare.dropna(subset=["meta_chm", "rh98"])

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.scatter(gedi_compare["rh98"], gedi_compare["meta_chm"], s=3, alpha=0.3)
ax.plot([0, 35], [0, 35], "r--", label="1:1 line")
ax.set_xlabel("GEDI rh98 (m)")
ax.set_ylabel("Meta CHM (m)")
ax.set_title("Meta CHM vs GEDI rh98 at shot locations")
ax.legend()

mae = mean_absolute_error(gedi_compare["rh98"], gedi_compare["meta_chm"])
rmse = np.sqrt(mean_squared_error(gedi_compare["rh98"], gedi_compare["meta_chm"]))
r2 = r2_score(gedi_compare["rh98"], gedi_compare["meta_chm"])
ax.text(0.05, 0.92, f"MAE: {mae:.2f} m\nRMSE: {rmse:.2f} m\nR²: {r2:.3f}",
        transform=ax.transAxes, fontsize=11, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()
print(f"N samples: {len(gedi_compare)}, MAE: {mae:.2f}m, RMSE: {rmse:.2f}m, R²: {r2:.3f}")